# UART 首次板级接收测试

本笔记本验证你自己的 UART 接收器和事件缓存，工作参数为 **100 MHz、115200 baud、8N1**。请按顺序逐格运行；在提示的位置切换到 Windows 发送数据，**不要直接 Run All**。

将以下四个文件上传到板上同一个文件夹 `uart_diag`：

- `uart_diag.bit` 和 `uart_diag.hwh`：从这次 Vivado 工程导出，二者名称相同、版本配套。
- `uart_diag.py`：本包提供的 Python 驱动。
- `02_uart_receive.ipynb`：本笔记本。

硬件接线按本包的使用说明完成。第一次只发送一个字节，再验证 64 字节短突发。FIFO 容量是 1024 条事件；本阶段不承诺 Python 可持续无损接收任意长度的满速数据。停止位错误会记录错误标志，但原接收器不会交付该错误帧的数据字节，因此表格中留空。


In [1]:
from pathlib import Path
from pynq import Overlay
from uart_diag import UARTDiagnostic, export_csv
from IPython.display import FileLink, display

for filename in ("uart_diag.bit", "uart_diag.hwh"):
    if not Path(filename).is_file():
        raise FileNotFoundError("请把 " + filename + " 上传到本笔记本所在文件夹")

ol = Overlay("uart_diag.bit")
if "axi_gpio_diag" not in ol.ip_dict:
    raise RuntimeError("硬件中没有 axi_gpio_diag，请确认上传的是本次 UART 工程的 bit/hwh")

rx = UARTDiagnostic(ol.axi_gpio_diag.mmio)
print("UART 硬件已加载。初始状态：")
print(rx.status())


UART 硬件已加载。初始状态：
{'count': 0, 'nonempty': False, 'overflow': False, 'ack': 0, 'framing_error': False, 'data_valid': False, 'data': None, 'raw': 2768240640}


## 第一次：发送一个字节 55

先确认 Windows 发送程序没有运行，然后执行下面的清空单元格。清空之后，再去 Windows 发送，回来读取。


In [2]:
rx.clear()
print("接收缓存已清空。现在去 Windows 发送十六进制 55，然后回来运行下一格。")


接收缓存已清空。现在去 Windows 发送十六进制 55，然后回来运行下一格。


### 在 Windows 上做这一步

在解压后的 `uart_board_stage1` 文件夹中打开命令提示符或 PowerShell。第一次使用先执行：

```bat
py -m pip install pyserial
```

然后执行（把 `COM7` 换成 **CH343 USB 转串口模块** 在设备管理器中的实际端口）：

```bat
py pc/send_uart.py --port COM7 --hex 55
```

开发板的 Micro-USB 控制台之前是 COM6；发送模块应使用它自己的端口。这里的 `55` 是一个值为 `0x55` 的字节，不是两个字符“5”。发送成功的提示只代表电脑已发出，下面的读取结果才用于判断板上接收是否正确。


In [3]:
events = rx.read_events(max_events=64)
state = rx.status()
print("本次读取事件数：", len(events))
print("序号  数据(hex)  停止位错误  数据有效  曾观察到溢出")
for event in events:
    print("{:>4}  {:>9}  {:>10}  {:>8}  {:>14}".format(
        event["index"], event["hex"] or "--",
        int(event["framing_error"]), int(event["data_valid"]),
        int(event["overflow"])))
print("FIFO 剩余条数：", state["count"])
print("本轮曾观察到溢出：", rx.overflow_seen)

passed = (len(events) == 1 and events[0]["data"] == 0x55
          and events[0]["data_valid"] and not events[0]["framing_error"]
          and not rx.overflow_seen and state["count"] == 0)
if passed:
    print("单字节测试通过：板上确实接收到了 55。")
elif not events:
    print("还没有收到事件。检查发送端口、TX/GND 接线以及是否先清空、后发送、再读取。")
else:
    print("结果与单字节 55 的预期不同，请保留本页输出并核查。")


本次读取事件数： 1
序号  数据(hex)  停止位错误  数据有效  曾观察到溢出
   1         55           0         1               0
FIFO 剩余条数： 0
本轮曾观察到溢出： False
单字节测试通过：板上确实接收到了 55。


In [ ]:
filename = export_csv(events, "uart_first_capture.csv")
print("已保存：", filename)
display(FileLink(filename))


## 可选第二步：64 字节连续接收

只有单字节测试通过后再执行。先运行下一格清空，然后回 Windows 发送：

```bat
py pc/send_uart.py --port COM7 --ramp
```

这次会发送 `00 01 02 ... 3F`，一共 64 字节。把 COM7 换成发送模块的实际端口。


In [5]:
rx.clear()
print("缓存已清空。现在在 Windows 执行 --ramp，再回来读取。")


缓存已清空。现在在 Windows 执行 --ramp，再回来读取。


In [6]:
events64 = rx.read_events(max_events=64)
state64 = rx.status()
actual = [event["data"] for event in events64]
expected = list(range(64))
print("接收数量：", len(events64))
print("接收内容：", " ".join(event["hex"] or "ERR" for event in events64))
print("FIFO 剩余条数：", state64["count"])
print("本轮曾观察到溢出：", rx.overflow_seen)
passed64 = (actual == expected
            and all(event["data_valid"] and not event["framing_error"] for event in events64)
            and not rx.overflow_seen and state64["count"] == 0)
print("64 字节测试通过。" if passed64 else "未达到 64 字节测试的预期，请保留输出。")
filename64 = export_csv(events64, "uart_64byte_capture.csv")
display(FileLink(filename64))


接收数量： 64
接收内容： 00 01 02 03 04 05 06 07 08 09 0A 0B 0C 0D 0E 0F 10 11 12 13 14 15 16 17 18 19 1A 1B 1C 1D 1E 1F 20 21 22 23 24 25 26 27 28 29 2A 2B 2C 2D 2E 2F 30 31 32 33 34 35 36 37 38 39 3A 3B 3C 3D 3E 3F
FIFO 剩余条数： 0
本轮曾观察到溢出： False
64 字节测试通过。


/home/xilinx/jupyter_notebooks/uart_diag/uart_64byte_capture.csv

## 保存与继续

- 按 **Ctrl + S** 保存代码和实际输出到 SD 卡；再用 **File → Download as → Notebook (.ipynb)** 下载到电脑备份。
- 点击上面的 CSV 链接下载实测记录。本阶段没有 FPGA 时间戳，CSV 不提供采样时刻。
- 下次重新启动板子后，从第一个代码单元格开始加载自己的硬件，再按“清空 → Windows 发送 → 网页读取”的顺序操作。
- 同一时刻只使用一个笔记本内核控制该 FIFO。若出现命令确认超时，记录报错，重新加载硬件并创建新驱动；不要继续发 POP 命令猜测状态。
- 这里的通过结论只覆盖已执行的单字节或 64 字节正常帧测试。停止位异常检测、更多波特率和长时间接收仍需各自的实测。

## 测试03：BREAK 引发停止位错误及接收恢复

### 测试条件
- 日期：2026-10-28
- 开发板：PYNQ-Z2
- FPGA设计时钟：100 MHz
- UART配置：115200 baud，8位数据，无校验，1位停止位
- 发送设备：CH343 USB转串口模块，COM5，3.3 V逻辑电平
- 接线：CH343 TX → PMODA 1；GND → PMODA 5
- 硬件文件：配套的 uart_diag.bit 和 uart_diag.hwh

### 测试步骤
1. 清空接收缓存。
2. Windows通过COM5调用 send_break(0.02)，请求发送一次BREAK。
3. BREAK结束后等待0.1秒，再发送正常字节55、AA。
4. 读取事件，检查错误标志、有效数据、缓存余量及溢出状态。

### 实测结果

| 事件序号 | 数据（十六进制） | 停止位错误 | 数据有效 |
|---|---|---|---|
| 1 | -- | 1 | 0 |
| 2 | 55 | 0 | 1 |
| 3 | AA | 0 | 1 |

- 共记录3条事件，顺序符合预期。
- 未观察到缓存溢出，读取后缓存为空。
- 本次测试通过。
- 原始记录：uart_break_recovery.csv

### 结论与说明
平台能够检测本次持续低电平BREAK引发的停止位错误，
并在信号恢复空闲后继续正确接收正常字节。

“--”表示该错误事件没有有效数据字节。
20毫秒是软件请求的BREAK持续时间，未进行波形实测；
0.1秒是人为设置的等待时间，不是测得的接收器恢复时间。
本次验证范围为BREAK异常与恢复，单帧坏停止位注入另行测试。

In [7]:
rx.clear()

{'count': 0,
 'nonempty': False,
 'overflow': False,
 'ack': 1,
 'framing_error': False,
 'data_valid': False,
 'data': None,
 'raw': 2768242688}

### Windows发送命令
在清空缓存后执行，命令结束后回到Jupyter读取。

```bat
py -c "import serial,time; s=serial.Serial('COM5',115200,timeout=1,write_timeout=2); s.send_break(0.02); time.sleep(0.1); s.write(bytes.fromhex('55 AA')); s.flush(); s.close(); print('BREAK + 55 AA sent')"
```

In [8]:
fault_events = rx.read_events(max_events=64)

for e in fault_events:
    print(
        e["index"],
        "数据:", e["hex"] or "--",
        "停止位错误:", int(e["framing_error"]),
        "数据有效:", int(e["data_valid"])
    )

actual = [
    (e["hex"], e["framing_error"], e["data_valid"])
    for e in fault_events
]
expected = [
    ("",   True,  False),  # BREAK 触发停止位错误
    ("55", False, True),   # 恢复后正常接收
    ("AA", False, True),
]

passed = (
    actual == expected
    and not rx.overflow_seen
    and rx.status()["count"] == 0
)
print("BREAK 异常与恢复：", "通过" if passed else "未通过")

display(FileLink(export_csv(fault_events, "uart_break_recovery.csv")))

1 数据: -- 停止位错误: 1 数据有效: 0
2 数据: 55 停止位错误: 0 数据有效: 1
3 数据: AA 停止位错误: 0 数据有效: 1
BREAK 异常与恢复： 通过


/home/xilinx/jupyter_notebooks/uart_diag/uart_break_recovery.csv